# Layer localization for selective unlearning — Colab runner

By **LinaGolan**.

Runs two localization methods (`gate` and `gate_margin`), intervenes on selected layers,
and measures WMDP-Bio accuracy against a retain set. The main grid compares eight layer
selections per method across five strengths; zero strength reuses the baseline.

For faster model inference in Colab, select an accelerator under **Runtime → Change runtime type**. Runtime depends on the environment. 

To inspect the included results without loading the model, get the code and install
dependencies, then jump to **7. Tables, intervals and figures**. No token or model inference is needed
for that analysis. Supplementary two- and four-layer runs are in section 8.


## 1. Check the runtime


In [ ]:
import torch
print("Accelerated inference available:", torch.cuda.is_available())
print("Saved-results analysis works without loading the model.")


## 2. Get the code

Clone the repository below, or clear `REPO_URL` to upload a zip instead.
Optionally mount Google Drive to preserve fresh results across session drops;
section 6 resumes from saved prediction files.


In [ ]:
USE_DRIVE = False   # set True to keep results across disconnects

import os, pathlib
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    workdir = pathlib.Path("/content/drive/MyDrive/unlearning_task")
    workdir.mkdir(parents=True, exist_ok=True)
else:
    workdir = pathlib.Path("/content")
os.chdir(workdir)
print("working in", workdir)

In [ ]:
REPO_URL = "https://github.com/LinaGolan/unlearning_task.git"

if not pathlib.Path("src/unlearning/experiment.py").exists():
    if REPO_URL:
        !git clone --depth 1 $REPO_URL repo
        os.chdir("repo")
    else:
        from google.colab import files
        uploaded = files.upload()               # pick a zip of the repository
        archive = next(iter(uploaded))
        !unzip -q -o "$archive"
        if not pathlib.Path("src/unlearning/experiment.py").exists():
            inner = sorted(pathlib.Path().glob("*/src/unlearning/experiment.py"))
            assert inner, "could not find src/unlearning/experiment.py in the upload"
            os.chdir(inner[0].parents[2])

assert pathlib.Path("src/unlearning/experiment.py").exists(), "repository not found"
os.environ["PYTHONPATH"] = str(pathlib.Path("src").resolve())
print("repository root:", pathlib.Path().resolve())
# Keep "results" to inspect/reuse the submitted predictions.
# Use "results_fresh/main" for a new experiment that preserves the submission.
RESULTS_DIR = "results"
FIGURES_DIR = "report/figures" if RESULTS_DIR == "results" else RESULTS_DIR + "/figures"


## 3. Install dependencies

Use Python 3.10 or newer.

`requirements.txt` selects packages by Python version. The historical run used
Python 3.12.11, torch 2.11.0+cu130 and transformers 5.16.1; those details are preserved
in `results/run.json`. The default installation does not recreate that environment
exactly, so a fresh model run may differ. Saved-prediction analysis does not load a model.


In [ ]:
!pip install -q -r requirements.txt
import torch, transformers, pyarrow, matplotlib
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("pyarrow     ", pyarrow.__version__)


## 4. Hugging Face access

`meta-llama/Llama-3.2-1B-Instruct` is gated, so a token with access is required. The WMDP and
MMLU data files are public and need no token.


In [ ]:
import getpass, os
os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token: ")

## 5. Prepare the questions (~1 min)

Downloads the ten pinned Parquet files, drops duplicate questions, and cuts disjoint
localization / development / test splits by SHA-256 rank. Prints the size of each split.


In [ ]:
!python -m unlearning prepare

## 6. Run the main experiment

Resumable: existing predictions in `results/predictions/` are reused. For a fresh run,
set `RESULTS_DIR = "results_fresh/main"` in section 2 before continuing. The model run,
analysis and readout cells all use that directory.
Changing a layer budget requires a different output directory.

The reduced example uses `results_reduced` so it cannot alter the submitted main results.


In [ ]:
!python -m unlearning run --out $RESULTS_DIR


In [ ]:
# Optional reduced experiment, kept separate from the submitted results.
# !python -m unlearning run --methods gate --strengths 0,0.5,1.0 --out results_reduced
# !python -m unlearning report --out results_reduced --figures results_reduced/figures


## 7. Tables, intervals and figures

Runs on CPU from `results/` alone. Regenerates comparison CSVs, detailed numerical
tables (including Appendix A), analysis JSON and three figures. The final
`Research_report.html` is maintained separately and is not overwritten.


In [ ]:
!python -m unlearning report --out $RESULTS_DIR --figures $FIGURES_DIR


In [ ]:
import csv
from pathlib import Path
for path in sorted(Path(RESULTS_DIR).glob("table_*.csv")):
    print(path)
    with path.open(encoding="utf-8", newline="") as handle:
        for row in csv.reader(handle):
            print(" | ".join(row))
    print()


In [ ]:
from IPython.display import Image, display
for name in ("localization.png", "strength.png", "tradeoff.png"):
    display(Image(f"{FIGURES_DIR}/{name}"))

In [ ]:
# Headline numbers, straight from the saved analysis.
import json
a = json.load(open(f"{RESULTS_DIR}/analysis.json", encoding="utf-8"))
b = a["baseline"]
print("baseline   WMDP %.2f%%   retain %.2f%%" % (100 * b["forget"]["accuracy"],
                                                  100 * b["retain"]["accuracy"]))
for method in a["methods"]:
    block = a["main"][method]
    print("\n%s  (alpha = %g)" % (method, block["alpha"]))
    for name, interval in block["intervals"]["forget"].items():
        retain = block["intervals"]["retain"][name]
        print("   %-20s dWMDP %+6.2f [%+6.2f, %+6.2f]   dRetain %+6.2f [%+6.2f, %+6.2f]" % (
            name, interval["drop_pp"], *interval["interval_95"],
            retain["drop_pp"], *retain["interval_95"]))
    for score, selection in a["localization"][method].get("selection", {}).items():
        print("   %s: best %s, runner-up %s, each half picks %s" % (
            score, selection["best_layer"], selection["runner_up"],
            selection["layer_chosen_by_each_half"]))

## 8. Two- and four-layer results (Table 6)

The saved supplementary runs use `gate` only. The next cell regenerates their
analyses from the saved predictions. Uncomment the model commands only when you intend to run
those experiments; existing predictions are cached. Use new output directories
for completely fresh runs.


In [ ]:
# !python -m unlearning run --methods gate --k 2 --strengths 0,0.5,1.0 --out results/k2
# !python -m unlearning run --methods gate --k 4 --strengths 0,0.5,1.0 --out results/k4
!python -m unlearning report --methods gate --out results/k2 --figures results/k2/figures
!python -m unlearning report --methods gate --out results/k4 --figures results/k4/figures


## 9. Save the results

Downloads the final HTML report, per-question predictions, localization scores,
analysis tables, figures and prepared data. Historical run metadata remains unchanged.


In [ ]:
import tarfile
from pathlib import Path
from google.colab import files
paths = {"results", RESULTS_DIR, FIGURES_DIR, "data", "Research_report.html"}
with tarfile.open("results.tgz", "w:gz") as bundle:
    for name in sorted(paths):
        path = Path(name)
        if path.exists() and not any(Path(parent) in path.parents for parent in paths if parent != name):
            bundle.add(path, arcname=name)
files.download("results.tgz")
